# 13.06 · Scaling & cost optimization

**Chapter:** Serving & Deployment · **Milestone:** M5

## Goals
- [ ] Measure latency properly: **TTFT**, inter-token latency, end-to-end, and percentiles (p50/p95), not averages
- [ ] Build a cost model: cost per request → per user → per month; API vs self-hosted break-even
- [ ] Apply the cost levers: smaller context, cheaper models, **cascades**, caching, batch APIs, output limits — and verify with evals
- [ ] Capacity planning with Little's law; autoscaling and cold starts

## Resources
- [Full Stack Deep Learning – LLM Bootcamp](https://fullstackdeeplearning.com/llm-bootcamp/) — free
- [Artificial Analysis](https://artificialanalysis.ai/) (independent latency/price/quality benchmarks of APIs) — free
- [FrugalGPT paper (cascades)](https://arxiv.org/abs/2305.05176) · [RouteLLM](https://arxiv.org/abs/2406.18665) — free
- Provider pricing pages + [OpenAI Batch API](https://platform.openai.com/docs/guides/batch) / [Gemini batch mode](https://ai.google.dev/gemini-api/docs/batch-mode) — free

The rule of this lesson: **every optimisation is an experiment**. Measure cost and latency *and* the eval score before and after.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from aieng import rag
from aieng.llm import available, client

## 1. Latency: measure the distribution
Users feel **TTFT** (first token) and the **tail**: the p95 request, not the average one. Measure with streaming, over enough calls.

In [ ]:
def measure(provider, prompt, n=10):
    c, model = client(provider)
    extra = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
    rows = []
    for _ in range(n):
        start, first, chunks = time.perf_counter(), None, 0
        try:
            stream = c.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], max_tokens=200,
                                               temperature=0.7, stream=True, **extra)
            for ch in stream:
                if ch.choices and ch.choices[0].delta.content:
                    first = first or time.perf_counter()
                    chunks += 1
        except Exception as e:  # free tiers rate-limit: record and continue
            print(f"{provider}: {type(e).__name__}"); time.sleep(5); continue
        end = time.perf_counter()
        if first:
            rows.append((first - start, end - start))
        time.sleep(1)  # stay under free-tier rate limits
    return np.array(rows)


prompt = "Explain the difference between latency and throughput in 3 sentences."
results = {p: measure(p, prompt) for p in ["ollama", "groq", "gemini"] if p in available()}
for p, r in results.items():
    if len(r):
        print(f"{p:7} TTFT p50 {np.percentile(r[:, 0], 50) * 1000:5.0f} ms  p95 {np.percentile(r[:, 0], 95) * 1000:5.0f} ms | "
              f"total p50 {np.percentile(r[:, 1], 50):4.1f}s p95 {np.percentile(r[:, 1], 95):4.1f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for p, r in results.items():
    if len(r):
        ax.scatter(r[:, 0] * 1000, r[:, 1], label=p, alpha=0.7)
ax.set_xlabel("TTFT (ms)"); ax.set_ylabel("end-to-end (s)"); ax.legend(); ax.set_title("each dot = one request"); plt.show()

Set a **latency budget** per feature and check p95 against it, e.g. chat: TTFT < 1 s, full answer < 8 s; autocomplete: total < 300 ms;
background jobs: minutes are fine (use batch APIs).
If a local model's p95 TTFT is far above its p50, look at the first request: that's Ollama loading the model into memory — a **cold start** (§4). The tail comes from queueing, cold starts, long outputs and provider load — so cap
`max_tokens`, set timeouts, and keep a fallback provider (11.02).

## 2. A cost model
Cost per request = input tokens × input price + output tokens × output price. Prices below are **illustrative** (USD per 1M tokens, they change
monthly: check the provider's page); the structure is what matters.

In [ ]:
PRICES = {  # name: ($ per 1M input, $ per 1M output) — illustrative only
    "small (flash-lite / 8B class)": (0.10, 0.40),
    "mid (flash / mini class)": (0.30, 2.50),
    "frontier (pro / opus class)": (3.00, 15.00),
}
hits = rag.search("Who was Abraham Lincoln?", k=5)
sources = "\n\n".join(f"[{i}] {t}" for i, _, t in hits)
tok_in = len(rag.PROMPT.format(sources=sources, question="Who was Abraham Lincoln?")) // 4  # ~4 chars/token rule of thumb
tok_out = 150
print(f"one RAG request ≈ {tok_in} input + {tok_out} output tokens\n")
for name, (pin, pout) in PRICES.items():
    per_req = (tok_in * pin + tok_out * pout) / 1e6
    print(f"{name:30} ${per_req * 1000:6.3f} per 1k requests | 10k users × 20 req/day: ${per_req * 10_000 * 20 * 30:9,.0f}/month")

**Self-hosting break-even**: a GPU costs per hour whether busy or idle. $/1M tokens = (GPU $/hour) / (tokens/s × 3600) × 1e6.

In [ ]:
for gpu, usd_h, tps in [("L4 24GB, 8B model, vLLM, batch", 0.80, 1500), ("H100 80GB, 70B FP8, vLLM, batch", 3.00, 3000),
                        ("H100, 70B, single user (no batching)", 3.00, 40)]:
    print(f"{gpu:38} ${usd_h / (tps * 3600) * 1e6:7.2f} per 1M output tokens (at 100% utilisation)")

Self-hosting only wins at **high, steady utilisation** (batching is what makes it cheap: compare the last two rows). At 10% utilisation the
price is 10× higher. For most products, API models are cheaper until volume is large or you need your own fine-tuned model.

## 3. Cost levers, measured
The biggest lever is usually **input tokens**: RAG prompts are mostly retrieved context. Experiment: answer accuracy vs number of passages.
Accuracy = the gold answer string appears in the reply (a cheap proxy metric from 08.04/10.04). We use the local model so it's free.

In [ ]:
c, MODEL = client("ollama")
EXTRA = {"reasoning_effort": "none"}
qs = [q for q in rag.eval_set() if len(q["answer"]) < 40][:30]


def ask(question, k):
    hits = rag.search(question, k)
    prompt = rag.PROMPT.format(sources="\n\n".join(f"[{i}] {t}" for i, _, t in hits), question=question)
    r = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0, max_tokens=150, **EXTRA)
    return r.choices[0].message.content, r.usage.prompt_tokens


def correct(reply, q):
    return q["answer"].strip().rstrip(".").lower() in reply.lower()


table = {}
for k in (1, 3, 8):
    start, ok, toks = time.perf_counter(), [], []
    for q in qs:
        reply, t = ask(q["question"], k)
        ok.append(correct(reply, q)); toks.append(t)
    table[k] = (np.mean(ok), np.mean(toks))
    print(f"k={k}: accuracy {np.mean(ok):.0%} | {np.mean(toks):5.0f} input tokens/request | {time.perf_counter() - start:.0f}s")

Now a **cascade**: try the cheap configuration first, escalate only when it fails a check. Here the check is free: the model said "I don't know"
(the prompt tells it to). Same idea with a cheap model → expensive model, where the check is a validator (JSON schema, 12.06),
a confidence score, or a small judge.

In [ ]:
ok, toks, escalated = [], [], 0
for q in qs:
    reply, t = ask(q["question"], 1)
    if "don't know" in reply.lower() or "do not know" in reply.lower():
        escalated += 1
        reply, t2 = ask(q["question"], 8); t += t2
    ok.append(correct(reply, q)); toks.append(t)
print(f"cascade k=1 → k=8: accuracy {np.mean(ok):.0%} | {np.mean(toks):5.0f} input tokens/request | escalated {escalated}/{len(qs)}")
print(f"always k=8      : accuracy {table[8][0]:.0%} | {table[8][1]:5.0f} input tokens/request")

Read the numbers: if the cascade keeps most of k=8's accuracy at a fraction of its tokens, that's a real saving. If "I don't know" is rare
even when the answer is wrong (overconfident model), the check is weak and the cascade silently loses accuracy — that's why you measure.

**The other levers** (in rough order of effort):
1. **Output limits**: `max_tokens`, "answer in ≤ 3 sentences" — output tokens cost 3–8× input tokens.
2. **Prompt caching**: put the static part (system prompt, instructions, few-shot) *first* so providers cache it (05.05): up to 90% off those tokens.
3. **Response caching**: exact + semantic cache for repeated questions (11.02).
4. **Batch APIs**: ~50% off for jobs that can wait up to 24 h (evals, enrichment, backfills).
5. **Smaller model** per task, verified with evals (10.x); **route** by difficulty (11.02); **fine-tune** a small model for narrow tasks (12.x).
6. **Self-host** quantized models (13.01–13.02) when volume is high and steady.

## 4. Capacity planning and autoscaling
**Little's law**: concurrent requests in flight = arrival rate × latency. It tells you how much concurrency your provider limits, your
API workers and your GPU batch size must support.

In [ ]:
for rps, latency_s in [(5, 3), (50, 3), (50, 8)]:
    print(f"{rps:3} req/s × {latency_s} s latency = {rps * latency_s:4} requests in flight")

- **Rate limits** (requests/min and tokens/min) from providers are a capacity limit: request increases early, spread load across providers/keys.
- **Autoscaling**: scale API containers on CPU / concurrency; scale GPU replicas on queue depth or KV-cache usage, not CPU.
- **Cold starts**: scale-to-zero saves money but a GPU replica can take 30 s – minutes to load weights. Keep a warm minimum for interactive
  traffic; bake models into images or use fast snapshot loading (Modal, RunPod FlashBoot).
- **Backpressure**: queue and shed load (429 + `Retry-After`) instead of letting latency explode for everyone.

## Exercises
1. Cut your app's cost per query by 50% without dropping its eval score. Keep a table: change → tokens → cost → score.
2. Re-run §1 with `max_tokens=50` vs `500`. How do p50 and p95 change? Which latency component moved?
3. Build a model cascade for the 12.x payments task: `smollm-payments` (local) first, escalate to Gemini when the JSON fails validation.
   Measure accuracy, escalation rate and cost on 200 examples, including the held-out templates.
4. Using real prices from two providers, compute the monthly cost of your M5 app at 1k, 10k and 100k daily users. At what point would a
   rented GPU with vLLM be cheaper (assume 30% utilisation)?

In [ ]:
# your code here